# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published on: {getattr(metadata, 'datePublished', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Citation: {getattr(metadata, 'citeAs', '')}")

print("\nSample metadata:")
pprint.pprint(metadata.to_json(), compact=True, depth=2)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This helps identify the structure of the data and what can be loaded.

In [ ]:
# List record sets in the dataset using their @id
print("Available RecordSets and their @id:\n")
record_sets = []
if hasattr(metadata, 'recordSet'):
    for recset in metadata.recordSet:
        recset_id = getattr(recset, '@id', None)
        recset_name = getattr(recset, 'name', None)
        print(f"  - @id: {recset_id} | name: {recset_name}")
        record_sets.append(recset_id)
else:
    print('No record sets found in the metadata.')

# Optionally show fields and columns in the first record set
if record_sets:
    print(f"\nFields in record set {record_sets[0]}:")
    for recset in metadata.recordSet:
        if getattr(recset, '@id', None) == record_sets[0]:
            fields = getattr(recset, 'field', [])
            for field in fields:
                fid = getattr(field, '@id', None)
                fname = getattr(field, 'name', None)
                dtype = getattr(field, 'dataType', None)
                print(f"    - @id: {fid}, name: {fname}, type: {dtype}")
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# This will dynamically load all available record sets into DataFrames
dataframes = {}

# For demonstration, we'll extract all available record sets
for recset_id in record_sets:
    print(f"Loading records from record set: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        if len(records) == 0:
            print("  - No records found.")
            continue
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"  - Columns: {df.columns.tolist()}")
        print(f"  - Sample rows:")
        display(df.head(3))
    except Exception as e:
        print(f"  - Could not load records from {recset_id}: {e}")

# For further steps, select a record set with data
main_record_set = None
for rec_id, df in dataframes.items():
    if not df.empty:
        main_record_set = rec_id
        break

print(f"\nSelected record set for EDA: {main_record_set}")
main_df = dataframes.get(main_record_set)
if main_df is not None:
    print(main_df.info())

## 4. Exploratory Data Analysis (EDA)
We'll perform exploratory data analysis by filtering, normalizing, and grouping numeric fields.

1. Select a numeric field to work with (e.g., coverage, peptide_count, mw)
2. Filter records based on a threshold
3. Normalize the numeric field
4. Group and aggregate by another field

In [ ]:
import numpy as np

# Identify a suitable numeric field. We'll search the DataFrame's columns for common numeric field names
candidate_fields = ['coverage', 'peptide_count', 'mw', 'abundance', 'pi', 'MW', 'Peptide Count', 'Coverage']
numeric_field = None
if main_df is not None:
    for field in candidate_fields:
        if field in main_df.columns:
            if pd.api.types.is_numeric_dtype(main_df[field]):
                numeric_field = field
                break

print(f"Selected numeric field: {numeric_field}")

# Perform analysis if we have a valid numeric field
if numeric_field is not None and main_df is not None:
    # Drop NAs for the selected field
    main_numeric = main_df.dropna(subset=[numeric_field])

    # Define a threshold for filtering (e.g., 10 for 'coverage' or 'peptide_count')
    threshold = main_numeric[numeric_field].quantile(0.75) if np.issubdtype(main_numeric[numeric_field].dtype, np.number) else 10

    filtered_df = main_numeric[main_numeric[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (Q3): {len(filtered_df)} rows")
    display(filtered_df.head(3))

    # Normalize the field
    field_norm = f"{numeric_field}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} statistics:\n{filtered_df[[numeric_field, field_norm]].describe()}")
    display(filtered_df[[numeric_field, field_norm]].head())

    # Identify a categorical/group field (e.g., sample, description, or other string field)
    cat_candidates = [col for col in main_df.columns if main_df[col].dtype == object and col != numeric_field]
    group_field = cat_candidates[0] if len(cat_candidates) > 0 else None
    print(f"Grouping by: {group_field}")

    if group_field is not None:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"Mean {numeric_field} by {group_field} (top groups shown):")
        display(grouped.head())
else:
    print("No numeric field found - skipping EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version. We'll use matplotlib and seaborn for simple plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and filtered_df is not None and not filtered_df.empty:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    
    plt.subplot(1,2,2)
    sns.histplot(filtered_df[field_norm], kde=True, color='orange')
    plt.title(f"Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field} (normalized)")
    
    plt.tight_layout()
    plt.show()
    
    # Boxplot by group if available
    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load Croissant metadata and records programmatically using the `mlcroissant` library
- Explore the dataset's record sets and field structure using their `@id`
- Extract and inspect tabular data for analysis
- Filter, normalize, and group data to prepare for downstream tasks
- Visualize numeric field distributions and group differences

The dataset provides detailed protein measurements from human mast cell extracellular vesicles, enabling in-depth proteomics analysis.

Next steps may include statistical testing, advanced visualization, or building models based on these protein features.